In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(r"C:\Users\RideW\Downloads\GeneralLexiconLinguisticCharacteristics.tsv", sep='\t')

In [ ]:
df.head()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore


score = 0
if "length" in df:    score += -zscore(df["length"].fillna(df["length"].median()))
if "syllables" in df: score += -zscore(df["syllables"].fillna(df["syllables"].median()))
if "subtlex" in df:   score +=  zscore(df["subtlex"].fillna(df["subtlex"].median()))
if "subtitles" in df: score +=  zscore(df["subtitles"].fillna(df["subtitles"].median()))
if "simple" in df:    score +=  zscore(df["simple"].fillna(df["simple"].median()))

df["simplicity"] = score
q_simple  = df["simplicity"].quantile(0.7)
q_complex = df["simplicity"].quantile(0.3)
df["bin"] = np.where(df["simplicity"] >= q_simple, "simple",
              np.where(df["simplicity"] <= q_complex, "complex", "neutral"))


In [ ]:
from huggingface_hub import login
login(os.environ["HF_TOKEN"])

In [ ]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"

tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tok.pad_token = tok.eos_token
tok.padding_side = "right"

In [ ]:
def single_token_id(word):
    ids = tok(word, add_special_tokens=False).input_ids
    return ids[0] if len(ids) == 1 else None

df["tok_id"] = df["word"].map(single_token_id)
df = df.dropna(subset=["tok_id"])

In [ ]:
import torch
V = tok.vocab_size
penalty = torch.zeros(V)          # 0 means no penalty
for _, r in df.iterrows():
    tid = int(r["tok_id"])
    if r["bin"] == "complex":
        penalty[tid] = 1.0        # complex gets penalty
    elif r["bin"] == "neutral":
        penalty[tid] = 0.5
    else:
        penalty[tid] = 0.0        # simple words are free
torch.save(penalty, "token_complexity_penalty.pt")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, AutoTokenizer
from peft import LoraConfig, get_peft_model

base = "meta-llama/Meta-Llama-3-8B-Instruct"   # or any causal LM you use
tok  = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, device_map="auto")

# LoRA
peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=["q_proj","k_proj","v_proj","o_proj"])
model = get_peft_model(model, peft_cfg)

token_complexity_penalty = torch.load("token_complexity_penalty.pt")

class SimplicityTrainer(Trainer):
    def __init__(self, lam=0.2, **kw):
        super().__init__(**kw)
        self.lam = lam
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs["labels"]                     # [B, T]
        out = model(**inputs)
        logits = out.logits                           # [B, T, V]
        ce = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            ignore_index=-100
        )
        with torch.no_grad():
            lab = labels.view(-1)
            mask = (lab != -100).float()
            safe = lab.clone()
            safe[safe == -100] = 0
            pen = token_complexity_penalty.to(logits.device)[safe]  # [B*T]
            pen = (pen * mask).sum() / (mask.sum() + 1e-8)
        loss = ce + self.lam * pen
        return (loss, out) if return_outputs else loss


args = TrainingArguments(
    output_dir="simplicity-ft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=2,
    bf16=True,
    logging_steps=50,
    save_steps=1000
)

trainer = SimplicityTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    lam=0.2
)
trainer.train()
